In [ ]:
import coffea
import coffea.hist as hist
import uproot
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np

import awkward as ak

from sklearn.cluster import DBSCAN

from coffea.nanoevents import NanoEventsFactory, NanoAODSchema,BaseSchema

In [ ]:
from collections import OrderedDict

# Load ntuples
Here we will load ntuples for signal and data (high-MET skim), where the clusters are already reconstructed and stored in the ntuples.

We will use these ntuples to plot some of the key variables that will used to veto background clusters

In [ ]:
fpath={}
tree ={}


# load high-MET skim data
path = '/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/data/'
fpath['data'] = path + 'Run2_displacedJetMuonNtupler_V1p17_Data2016_Data2017_Data2018-HighMET_goodLumi.root'

mass = [40]
OLD_CTAU = np.array([1000, 10000, 100000])#in mm

path = '/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/signal/nocuts/'

for m in mass:
    for ct in OLD_CTAU:
        key = 'MC_'+str(m)+'_'+str(ct)                       
        fpath[key] = path +  'ggH_HToSSTobbbb_MH-125_MS-40_ctau-'+str(ct)+'_137000pb_weighted.root'
        
        
for k,v in fpath.items():
    tree[k] = NanoEventsFactory.from_root(v,
                                   schemaclass=BaseSchema,
                                   treepath="MuonSystem"
                                  ).events()

In [ ]:
# See what's contained in the tree dictionary
tree

# Trigger selection

Trigger selection is the beginning of any CMS analysis. 

We are using `MET > 200 GeV` due to the lack of dedicated trigger on the MDS object. 

As a result, in order for signal events to pass the large MET trigger
 - the Higgs is recoiled against a high pT jet
 - When the LLPs decays beyond calorimeters, the decay
 - Since the Higgs has a high pT, the MET will be large
 
You will try to confirm this for yourself first.

 - Use the `met` branch to access the reconstructed MET
 - Use the `gHiggsPt` branch to access the generated Higgs pT  

# Ex: Plot the MET and Higgs pT for signals

 - Focus on the region beyond 200 GeV
 - Q: What do you observe?

In [ ]:
h= hist.Hist("Events",
              hist.Cat("var","Variable"), 
              hist.Bin("pT", "MET/Higgs pT[GeV]", 50, 200, 1200)
             )

sample = "MC_40_1000"
# Fill the code
h.fill(var = "MET", pT = tree[sample].met )
h.fill(var = "Higgs pT", pT = tree[sample].gHiggsPt )
        
ax = hist.plot1d(h,density=True)
ax.legend(title = sample)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary") 

## Explanation: MET matches the Higgs pT

`tree[sample].met` is the reconstructed missing transverse momentum and `tree[sample].gHiggsPt` is the generator-level pT of the Higgs, both read straight from the coffea/NanoEvents tree (so they are awkward arrays with one entry per event, which `hist.fill` accepts directly).

The two shapes lie on top of each other. That is exactly the topology of the benchmark model: the Higgs is produced back-to-back against one hard ISR jet, and if the two LLPs escape the calorimeters before decaying, the Higgs momentum is carried away invisibly, so MET ≈ pT(Higgs).

Physics meaning for the trigger: we have no dedicated MDS trigger in Run 2 so we fire on `MET > 200 GeV`, and because MET follows the Higgs pT this trigger only accepts the tail of the distribution where the Higgs is very boosted. That is why the signal efficiency is so small (about 1%, next cell).

Pitfalls: (1) the plot is drawn with `density=True`, so only shapes can be compared, not yields; (2) the axis starts at 200 GeV, everything below it goes to the underflow, which is why the soft tail is not visible here.

# Ex: Trigger efficiency for signals

In [ ]:
plt.style.use(hep.style.CMS)  

In [ ]:
h_MET= hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("met", "MET[GeV]", 50, 0, 500)
             )

for m in mass:
    for ct in OLD_CTAU:
        # Fill the histogram h_MET with all the signals
        sample = 'MC_'+str(m)+'_'+str(ct)
        h_MET.fill(sample=sample, met=tree[sample].met)
        
ax = hist.plot1d(h_MET,density=True)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What is the signal efficiencies for MET>200 GeV?

 - Use the following function `integral` the dense axes from `thres_lo` to infinity
 - Use the function `integral_low` the dense axes from lower bound to thres_high (for later parts)

In [ ]:
#  Integral the dense axes from thres_lo to infinity
def integral(h,sample,thres_lo):
    
    name = h.dense_axes()[0].name
    
    numer = h.integrate("sample",sample).integrate(name,slice(thres_lo,None),overflow="over").values()[()]
    denom = h.integrate("sample",sample).sum(name,overflow="over").values()[()]    
    
    return numer/denom

# Integral the dense axes from lower bound to thres_high
def integral_low(h,sample,thres_high):
    
    name = h.dense_axes()[0].name

    numer = h.integrate("sample",sample).integrate(name,slice(None,thres_high)).values()[()]
    denom = h.integrate("sample",sample).sum(name,overflow="over").values()[()]    

    return numer/denom

In [ ]:
# Choose from the following bin edges
h_MET.axis("met").edges()

In [ ]:
for m in mass:
    for ct in OLD_CTAU:
        sample = 'MC_'+str(m)+'_'+str(ct) 
        # Fill the code
        eff = integral(h_MET, sample, 200)
        print("sample = %s, eff(MET>200) = %.5f "%(sample, eff))

## Explanation: trigger efficiency of the signals

`integral(h, sample, thres_lo)` (defined above) divides the histogram content from `thres_lo` to infinity, including overflow, by the total content, so `integral(h_MET, sample, 200)` is exactly the fraction of signal events with MET > 200 GeV, i.e. the efficiency of our MET > 200 GeV trigger. 200 GeV is a real bin edge of `h_MET` (50 bins from 0 to 500 GeV, so 10 GeV wide), which is why the cell just above prints `h_MET.axis('met').edges()`.

The numbers come out around 1e-2 or smaller. That is expected: the LLP pair from a 125 GeV Higgs is usually soft, so we only keep the rare events where an energetic ISR jet kicks the Higgs to a very large pT and the LLPs fly out of the detector before decaying.

Pitfalls: (1) the efficiency is a ratio of *weighted* histogram contents, so it is only meaningful because the signal samples are already `..._weighted.root`; (2) a longer-lived LLP (larger ctau) decays further out, so it looks more invisible and passes the MET trigger more often - the efficiency is ctau dependent and must be kept per sample in the limit setting.

# Event selections

The event selections for this search are kept at minimal to be as model independent as possible. 

The full event-level selections are 
 - MET trigger 
 - offline MET > 200 GeV
 - MET filters (make sure MET is not badly reconstructed)

For the ISR jet, the selections are
 - pT > 30 GeV
 - abs($\eta$) < 2.4
 - Pass tight jet Id

In [ ]:
def select_events(events,isData=False):
    
    ########### SELECTION: CLUSTERS ############
    ### At least 1 cluster (decay in CSC and matched to LLP for signal)
    ###
    if isData:
        sel_clusters = (events.cscRechitClusterSize)>0
    else:
        sel_clusters = (events.cscRechitCluster_match_gLLP==True) & (events.cscRechitCluster_match_gLLP_csc==True)

    ########### SELECTION: JETS ############
    sel_jet = (events.jetPt > 30) &\
              (abs(events.jetEta)<2.4) &\
              (events.jetTightPassId==True)
    
    ########### SELECTION: events ############
    # at least 1 selected cluster and jets
    # and pass MET selections
    sel_ev =  (events.METTrigger==True) &\
              (events.met>200) &\
              (events.metFilters==True) &\
              (ak.sum(sel_jet,axis=1)>=1) &\
              (ak.sum(sel_clusters,axis=1)>0)   
    
    return sel_clusters, sel_jet, sel_ev

# Cluster level selections

Aside from the MDS shower, there are not many handles in the events to reject background events.

Afterall, we only expect the shower, MET and an ISR jet in the signal events. 

Therefore, we will focus on using the MDS object features to enhance the signal to background ratio.  

First, we need to remove objects from the SM that could also create showers in the muon system. 

 - Punch-through jets
 - high pT muons


The dominant background from the main collision comes from punch-through jets that are not fully contained in the calorimeters and high pT muons that could create bremsstrahlung showers in the muon detectors. 

To remove those background, we reject clusters by matching them to reconstructed jets and muon. 

The pT of the jet/muon that the cluster is matched to are saved in objects called `cscRechitClusterJetVetoPt` 
and `cscRechitClusterMuonVetoPt`. 

In this exercise you will plot the two variables for signal and background and apply a selection on the two variables to remove clusters from jets and muons.

# Ex1 - plot jet veto and muon veto

 - Access the jet veto pT with `cscRechitClusterJetVetoPt`
 - Access the muon veto pT with `cscRechitClusterMuonVeto`

In [ ]:
nBins    = 50
bin_lo   = 0 
bin_high = 500 

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("pt", "Jet veto pT[GeV]", nBins, bin_lo, bin_high)
             )

for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # get the Jet veto Pt
    VetoPt = tree[sample].cscRechitClusterJetVetoPt
    
    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    #
    # Apply the cluster and event selections onto VetoPt
    VetoPt = VetoPt[sel_clusters][sel_ev]
    
    # Select the VetoPt of the largest cluster in the event
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    VetoPt = ak.fill_none(ak.firsts(VetoPt,axis=1),-1 ) 
    
    h.fill(sample=sample,pt=VetoPt)

## Explanation: Ex1 jet veto pT

`cscRechitClusterJetVetoPt` is a cluster-level (nested) awkward array: one list of values per event. `VetoPt[sel_clusters]` keeps only the clusters we want (for data: any cluster, for signal: only clusters matched to an LLP that decayed in the CSC), and `[sel_ev]` then keeps only the events that pass the event level selection (MET trigger, MET > 200 GeV, MET filters, at least one tight ISR jet). Masking in this order is the awkward/numpy style used everywhere in this notebook.

`ak.firsts(VetoPt, axis=1)` then takes the first (largest) cluster of every event, and `ak.fill_none(..., -1)` gives -1 to events that have no surviving cluster; -1 falls in the underflow of the 0-500 GeV axis, so the plot only shows events that really have a cluster.

What the next plot should show: signal sits at very small jet veto pT (the LLP shower has no jet pointing at it, so nothing is matched and the value is 0), while data has a large tail up to several hundred GeV from punch-through jets. The `density=True` normalisation hides the huge difference in event counts, so look at the shapes, not the heights.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-3,0.1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies Jet veto pT < 10 GeV?

In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 10)
    print("sample = %s, eff(VetoPT<10) = %.5f "%(sample, eff))

## Explanation: efficiency of Jet veto pT < 10 GeV

`integral_low(h, sample, 10)` returns the fraction of events in `h` (the jet veto histogram we just filled) with a value below 10 GeV, which is the efficiency of the cut `cscRechitClusterJetVetoPt < 10`. 10 GeV is a bin edge of the 0-500 GeV, 50-bin axis, so the number is not smeared by partial bins.

Expected behaviour: signal keeps almost 100% (its clusters are not matched to jets), data keeps much less because the punch-through jet background is removed. Note the efficiency is computed with respect to events that have a surviving cluster, since the -1 entries sit in the underflow and are not counted in the denominator.

Pitfall: the veto is applied to the *largest* cluster only (`ak.firsts`), so this number is an event-level efficiency for the leading cluster, not a per-cluster efficiency.

# Plot muon veto 

In [ ]:
# Think about the bin ranges and number of bins you want to in the histogram before plotting
nBins    = 50
bin_lo   = 0
bin_high = 500

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("pt", "Muon veto pT[GeV]", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # get the Muon veto Pt
    VetoPt = tree[sample].cscRechitClusterMuonVetoPt
    
    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    #
    # Apply the cluster and event selections onto VetoPt
    VetoPt = VetoPt[sel_clusters][sel_ev]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    VetoPt = ak.fill_none(ak.firsts(VetoPt,axis=1),-1 ) 
    
    h.fill(sample=sample,pt=VetoPt)

## Explanation: Ex1 muon veto pT

Same recipe as the jet veto, only the branch changes to `cscRechitClusterMuonVetoPt`, the pT of the reconstructed muon matched to the cluster. I kept the axis at 0-500 GeV with 50 bins (10 GeV wide) so that the analysis working point, 20 GeV, is a bin edge and the efficiency in the next cell is exact.

What to look for: signal again peaks at 0 (nothing recoils against a cluster made by an LLP that decayed outside the calorimeters), while data has a long high-pT tail from muon bremsstrahlung. The high-mass muon-brem component is exactly what the `cscRechitClusterMuonVetoPt < 20` cut throws away.

Pitfall: a small fraction of signal clusters do get matched to a low-pT jet or muon by accident (the matching cone can catch an unrelated ISR jet), which is the small non-zero population of signal above the veto thresholds, i.e. the signal efficiency loss of the cut.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-5,0.1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies Muon veto pT < 20 GeV?

In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 20)
    print("sample = %s, eff(Muon VetoPT < 20) = %.5f "%(sample, eff))

## Explanation: efficiency of Muon veto pT < 20 GeV

`integral_low(h, sample, 20)` divides the content of `h` below 20 GeV by the total content, i.e. it is the efficiency of `cscRechitClusterMuonVetoPt < 20`. The threshold 20 GeV is higher than the jet one (10 GeV) because a muon that can make bremsstrahlung photons in the muon system is typically harder to reject with a low threshold, and because the muon momentum scale has bigger tails than the jet energy.

Signal efficiency should be close to 1, so the two vetoes together cost very little signal while removing the bulk of the in-collision background.

Pitfall: this histogram `h` is the muon veto one, so the number is only meaningful if the plot cell above was run last; the jet veto efficiency cell reuses the same variable name `h`.

# Ex2 - plot number of ME11 and ME12 hits in the cluster

Additionally, punch-through jets or muon bremsstrahlung showers *might not get reconstructed as jets and muons*.

To fully remove these background, we remove clusters that have hits in the first CSC stations (ME11/ME12) that have little shielding in front.

In this exercise, you will plot the number of ME11/ME12 hits in clusters for signal and background and apply a selection on the two variables to remove clusters from jets and muons.

## plot number of ME11 and ME12 hits in cluster

 - Access the number of chamber hits in the cluster with the following variables:
   - `cscRechitClusterNRechitChamberPlus11`
   - `cscRechitClusterNRechitChamberMinus11`
   - `cscRechitClusterNRechitChamberPlus12`
   - `cscRechitClusterNRechitChamberMinus12`

In [ ]:
nBins    = 50
bin_lo   = 0
bin_high = 50

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("nME11_12", r"$N_{Me11/12 hits}$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Compute nME11_12
    nME11_12 = (tree[sample].cscRechitClusterNRechitChamberPlus11  +
                tree[sample].cscRechitClusterNRechitChamberMinus11  +
                tree[sample].cscRechitClusterNRechitChamberPlus12  +
                tree[sample].cscRechitClusterNRechitChamberMinus12)
    
    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    # Apply the cluster and event selections 
    nME11_12 = nME11_12[sel_clusters][sel_ev]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    nME11_12 = ak.fill_none(ak.firsts(nME11_12,axis=1),-1 ) 
    
    h.fill(sample=sample,nME11_12=nME11_12)

## Explanation: Ex2 hits in ME11/ME12

The four branches store the number of reconstruct hits in ME1/+1, ME1/-1, ME1/+2 and ME1/-2 separately, so the total number of hits in the two least-shielded chambers is just their sum (awkward adds nested arrays element by element). Then the same `[sel_clusters][sel_ev]` mask keeps the right clusters in the right events, and the axis 0-50 with 50 bins gives one bin per hit so `nME11_12 == 0` is a single bin.

What to look for: background clusters have a large population at 1, 2, 3 ... hits in ME11/ME12 because a punch-through jet or a bremsstrahlung shower enters the muon system from the inside and crosses the first stations on its way out. Signal clusters come from LLPs that decay deep in the muon system, often only in ME2-ME4 or in the endcaps beyond ME1, so they mostly sit at 0 hits.

Pitfalls: (1) an LLP that decays early, in ME1/3 or ME2, can still leave hits in ME11/ME12, so this veto is not free - it costs signal efficiency; (2) pileup clusters are not affected by this veto in the same way as punch-through, which is why the out-of-time region still needs the ABCD estimate later on.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-4,1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for nME11_12 ==0?

In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 1)
    print("sample = %s, eff(nME11_12 ==0) = %.5f "%(sample, eff))

## Explanation: efficiency of nME11_12 == 0

Because the histogram is one bin per hit, 'exactly zero hits' is the same as 'below 1 hit', so `integral_low(h, sample, 1)` is the efficiency of requiring `nME11_12 == 0`.

This is the 'active veto' of the analysis: it catches punch-through jets and muon bremsstrahlung that were too soft or too badly reconstructed to be rejected by the jet and muon vetoes. It is the selection with the largest signal efficiency loss among the cluster vetoes, since LLPs that decay in the innermost muon chambers also see ME11/ME12.

Pitfall: the numerator counts events whose leading cluster has 0 hits in ME11/ME12; the -1 fill value (no cluster at all) is in the underflow and cancels out of the ratio, so this number does not tell you anything about how many events have a cluster.

# Ex3 - plot cluster eta

After we removed punch-through jets and muon brems, we observed that there are a lot more backgorund in higer $\eta$ region, where the muon reconstruction efficiency is lower and more pileup particles are present to create clusters.

In this exercise you will plot the cluster $\eta$ for signal and background and apply a selection on the variable to remove clusters from high $\eta$ region.

In [ ]:
nBins    = 50
bin_lo   = 0
bin_high = 2.5

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("eta", r"cluster $\eta$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get cluster Eta
    clsEta = abs(tree[sample].cscRechitClusterEta)
    
    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    # Apply the cluster and event selections 
    clsEta = clsEta[sel_clusters][sel_ev]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    clsEta = ak.fill_none(ak.firsts(clsEta,axis=1),-1 ) 
    
    h.fill(sample=sample,eta=clsEta)

## Explanation: Ex3 cluster eta

I take `abs(cscRechitClusterEta)` because the analysis works on |eta| (the two muon endcaps are mirror images) and because the histogram axis then runs from 0 to 2.5 with 0.05 wide bins, so the working point |eta| < 2 is a bin edge and the efficiency in the next cell is a clean `integral_low`.

What to look for: the data distribution grows towards the edges of the CSC acceptance (|eta| ~ 2.4), where there are more pileup interactions per unit eta and less shielding and coverage in front of the chambers. Signal is much flatter, reflecting the roughly uniform decay position of the LLPs inside the muon system acceptance.

Pitfall: this plot is made *before* the jet/muon vetoes are combined with the eta cut, so part of the forward rise is still punch-through background that the earlier vetoes already remove; the eta cut is really about the remaining pileup rate.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-4,10)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $abs(\eta)<2$ ?

In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 2)
    print(r'sample = %s, eff(abs($\eta$)<2 ==0) = %.5f '%(sample, eff))

## Explanation: efficiency of abs(eta) < 2

Since the previous cell filled the histogram with |eta|, the cut |eta| < 2 is again a simple `integral_low(h, sample, 2)`.

Signal keeps a high efficiency (the LLP decays inside the muon system, and the CSC covers |eta| up to about 2.4), while data loses a large fraction of its clusters, which is what we want: the forward region is where pileup clusters are most numerous and where the muon system has the least material in front.

Pitfall: cutting too hard on eta shrinks the acceptance for long-lived LLPs that decay at large radius in the endcap disks, and it also removes the most populated part of the out-of-time validation region, which costs background statistics for the ABCD method in the next episode.

# Ex4 - plot cluster time (after vetos are applied)

The remaining background clusters after punch-through jet and muon brem showers from the main collision are removed, are from low pT particles in pileup events.

To verify this, you will plot the cluster time for signal and background, after applying the vetoes, in this exercise to check for any out-of-time pileup contributions in data.


In [ ]:
nBins    = 120
bin_lo   = -30
bin_high = 30

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("time", r"cluster time[ns]", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get Variables
    clsTime  = tree[sample].cscRechitClusterTime
    jetVeto  = tree[sample].cscRechitClusterJetVetoPt
    muonVeto = tree[sample].cscRechitClusterMuonVetoPt
    clsEta   = abs(tree[sample].cscRechitClusterEta)
    
    # Apply the following selections:
    # - cluster and event selections 
    # - JetVeto < 10 GeV
    # - MuonVeto < 20 GeV
    # - cluster eta < 2
    sel_cls = sel_clusters & sel_ev & (jetVeto < 10) & (muonVeto < 20) & (clsEta < 2)
    
    # Try running without Jet and muon Veto as well!
    clsTime = clsTime[sel_cls]
    # without the vetoes: clsTime = clsTime[sel_clusters & sel_ev]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    clsTime = ak.fill_none(ak.firsts(clsTime,axis=1),-999 ) 
    
    h.fill(sample=sample,time=clsTime)

## Explanation: Ex4 cluster time and the signal region

All four variables are cluster-level awkward arrays, so they can be combined with `&` into one mask. `sel_ev` is a per-event (one value per event) boolean, and awkward automatically broadcasts it onto the cluster lists, which is why `sel_clusters & sel_ev & (jetVeto < 10)` is legal. `fill_none(..., -999)` pushes events without a surviving cluster into the underflow of the -30 to 30 ns axis, and I chose 120 bins (0.5 ns wide) so that both -12.5 ns and -5 ns are bin edges.

What the plot shows (Discussion 4.4/4.5): the data is not smooth, it has peaks every 25 ns. Those are pileup clusters from the other proton-proton collisions in the same and in neighbouring bunch crossings, and the LHC bunches cross every 25 ns, so their hit times are quantised in 25 ns steps relative to the hard scatter. Signal (delayed LLP decays) is spread over the whole range instead.

With the jet and muon vetoes ON, the huge in-time peak around 0 ns largely disappears (that peak is punch-through and bremsstrahlung from the main collision); the out-of-time peaks stay, which proves they are pileup and not detector effects. The analysis then takes the window -12.5 ns < t < -5 ns as the signal region: it lies between the bunch crossings, so prompt activity cannot land there, while a delayed LLP decay can.

Pitfall: keep the -999 fill value in mind when you read the ratio efficiencies, and remember the commented line above lets you re-run the cell without the vetoes to check Discussion 4.5 by eye.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $-5<t<12.5$ ns ?

In [ ]:
def integral_range(h,sample,thres_lo,thres_high):
    
    name = h.dense_axes()[0].name

    numer = h.integrate("sample",sample).integrate(name,slice(thres_lo,thres_high)).values()[()]
    denom = h.integrate("sample",sample).sum(name,overflow="over").values()[()]    

    return numer/denom

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_range(h, sample, -12.5, -5)
    print(r'sample = %s, eff( -5 < t < 12.5 ns) = %.5f '%(sample, eff))

## Explanation: efficiency of the signal region window

`integral_range(h, sample, -12.5, -5)` integrates the time histogram between -12.5 ns and -5 ns and divides by the total, i.e. the fraction of clusters that lands in the signal region. -12.5 and -5 are bin edges of the 0.5 ns axis, so nothing is smeared.

Note the heading of this question is printed as `-5 < t < 12.5` but that is a typo in the material: the episode text defines the signal region as `-12.5 ns < t < -5 ns` (an *early* window), and that is what I fill in. Filling the literal `(-5, 12.5)` would measure the in-time peak instead, and for data it would also unblind the signal region.

Pitfall: the efficiency is smaller than one would naively expect for prompt objects because the signal region covers only 7.5 ns out of the 25 ns between bunch crossings, about 30% of the phase space - that is the price of a background-free window.

# Q: Can you suggest an out-of-time validation region based on the above plot?

A good validation region needs to have:
 - No (or at least small) signal conteminations (i.e. signal depleted) 
 - Similar background composition as your signal-rich region

## Answer: the early out-of-time (OOT) validation region, t < -12.5 ns

Looking at the vetoed data time distribution, the clean choice is everything earlier than the signal region: `cluster time < -12.5 ns`, i.e. the pileup peak of the bunch crossing before the previous one (the -25 ns peak and its tails). It is the same kind of sideband the signal region is: both windows sit between the peaks, so the clusters there are out-of-time pileup in both cases.

Why it satisfies the two requirements: (1) it is signal depleted - a genuine LLP decay would have to be very early relative to the hard scatter, which the exponential decay time distribution essentially never does, and the peak of the signal time distribution sits at positive (delayed) times, so almost no signal falls below -12.5 ns; (2) the background composition is the same as in the signal region, because both regions are populated by the same low-pT pileup clusters, and the jet/muon/eta vetoes do not touch them (they are not matched to hard objects), so the shape of N_hits and |dphi(cluster,MET)| is unchanged.

Pitfall: an even earlier region (t < -30 ns) would be emptier of signal but also too small in statistics, and a window that touches a bunch-crossing peak (e.g. -20 < t < -15) would mix in a different background population.

# Ex5 - plot $\Delta\phi\mathrm{(cluster, MET)}$ and $N_{hits}$ in Out-of-time Validation Region (final discriminating variables)


The final discriminating variables that we will use to extract the signal and estimate background are 
 - the number of hits in the cluster ($N_{\text{hits}}$) and
 - the azimuthal angle between the cluster and MET ( $\Delta\phi\text{(cluster, MET)}$).

The background estimation method will be described in more detail in the next episode.

In this exercise, we will just plot the distributions of the two variables, to understand the shape of the two variables.

## Plot data only in OOT region to avoid unblinding!

## Implement the OOT selection you came up with above!

## plot $N_{hits}$

In [ ]:
nBins    = 50
bin_lo   = 0
bin_high = 500

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("nhits", r"$N_{hits}$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get Variables
    nhits    = tree[sample].cscRechitClusterSize
    clsTime  = tree[sample].cscRechitClusterTime
    jetVeto  = tree[sample].cscRechitClusterJetVetoPt
    muonVeto = tree[sample].cscRechitClusterMuonVetoPt
    clsEta   = abs(tree[sample].cscRechitClusterEta)

    
    # Only plot Out-of-Time(OOT) region for data
    if isData:
        sel_time = clsTime < -12.5
    else:
        sel_time = clsTime > -12.5   
        
    # Apply the following selections:
    # - cluster and event selections 
    # - JetVeto < 10 GeV
    # - MuonVeto < 20 GeV
    # - cluster eta < 2
    # - OOT selection for data; in-time selection for data
    sel_cls = sel_clusters & sel_ev & (jetVeto < 10) & (muonVeto < 20) & (clsEta < 2) & sel_time
    nhits = nhits[sel_cls]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    nhits = ak.fill_none(ak.firsts(nhits,axis=1),-1 ) 
    
    h.fill(sample=sample,nhits=nhits)

## Explanation: Ex5 N_hits in the out-of-time region

`cscRechitClusterSize` is the number of reconstruct hits in the cluster (N_hits) and the axis 0-500 with 50 bins makes 130, the analysis working point, a bin edge.

The OOT selection is written as an if/else so that data and signal are never shown in the same time window: for data we take `clsTime < -12.5` ns (the early out-of-time validation region, where we are allowed to look), for signal we take the complementary `clsTime > -12.5` ns, i.e. everything the OOT region excludes (the signal region plus the in-time region), so the signal shape is not artificially truncated. The data side is the region the ABCD closure test will be done in during the next episode.

The rest is the same cluster-level mask as in Ex4 (`sel_clusters & sel_ev` plus the three vetoes), broadcast onto the cluster level, and `fill_none(..., -1)` puts events without a surviving cluster in the underflow.

What to look for: data in the OOT region falls steeply with N_hits, most clusters have only a handful of hits, while the signal distribution sits at hundreds of hits because an LLP decay produces a genuine hadronic shower in the chambers. That is why N_hits > 130 enriches signal.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-4,0.1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $N_{hits}>130$ ?


In [ ]:
for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral(h, sample, 130)
    print(r'sample = %s, eff($N_{hits}>130$) = %.5f '%(sample, eff))

## Explanation: efficiency of N_hits > 130

`integral(h, sample, 130)` integrates from 130 to infinity (overflow included) and divides by the total, so it is the efficiency of requiring more than 130 hits. The histogram here is the OOT/in-time one filled just above, so the number quoted for data is the fraction of *out-of-time* clusters that pass, which is exactly the side we need to keep small.

Signal efficiency is high (an LLP shower easily produces hundreds of hits), data efficiency is tiny, so this cut is the main handle that separates signal from pileup.

Pitfall: the requirement is a per-cluster one but the histogram stores the leading cluster only, so an event whose largest cluster fails but whose second cluster passes is counted as failing. Also, pushing the threshold higher keeps improving purity but empties the OOT validation region, and the ABCD estimate needs enough events in every bin.

## plot abs($\Delta\phi\mathrm{(cluster, MET)}$)

In [ ]:
nBins    = 60
bin_lo   = 0
bin_high = 3.0

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("dphi", r"$\Delta\phi(cls,MET)$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get Variables
    dphi     = abs(tree[sample].cscRechitClusterMet_dPhi)
    nhits    = tree[sample].cscRechitClusterSize
    clsTime  = tree[sample].cscRechitClusterTime
    jetVeto  = tree[sample].cscRechitClusterJetVetoPt
    muonVeto = tree[sample].cscRechitClusterMuonVetoPt
    clsEta   = abs(tree[sample].cscRechitClusterEta)

    
    # Only plot Out-of-Time(OOT) region
    if isData:
        sel_time = clsTime < -12.5
    else:
        sel_time = clsTime > -12.5    
        
    # Apply the following selections:
    # - cluster and event selections 
    # - JetVeto < 10 GeV
    # - MuonVeto < 20 GeV
    # - cluster eta < 2
    # - OOT selection for data; in-time selection for data
    sel_cls = sel_clusters & sel_ev & (jetVeto < 10) & (muonVeto < 20) & (clsEta < 2) & sel_time
    
    dphi = dphi[sel_cls]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    dphi = ak.fill_none(ak.firsts(dphi,axis=1),-999 ) 
    
    h.fill(sample=sample,dphi=dphi)

## Explanation: Ex5 abs(delta phi(cluster, MET)) in the out-of-time region

`cscRechitClusterMet_dPhi` is the azimuthal angle between the cluster direction and the MET direction; I take `abs()` of it and plot 0 to 3.0 with 60 bins (0.05 wide) so the working point 0.75 is a bin edge (angles between 3.0 and pi end up in the overflow, which the efficiency functions below count in the denominator). The branch is used again in exactly this form in `ABCD_validation.ipynb`.

The selection chain is identical to the N_hits cell, so the two histograms describe the same OOT data sample: data below -12.5 ns, signal above it, all with the jet, muon and eta vetoes and the event selection applied.

What to look for (Question 4.1): the signal distribution peaks towards 0 because the LLP cluster is emitted along the Higgs direction and the Higgs *is* the MET, so the cluster points at the MET. The data distribution in the OOT region is flat: a pileup cluster from another interaction has no reason to be aligned with the MET of the hard scatter.

Pitfall: this is why the two variables are chosen for the ABCD method - flatness in delta phi together with a steeply falling N_hits means the two are independent for background - but the independence must be checked, not assumed, which is what the correlation and closure tests of the next episode do.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $abs(\Delta\phi(cls,MET)<0.75)$ ?


In [ ]:
for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 0.75)
    print(r'sample = %s, eff($abs(\Delta\phi(cls,MET)<0.75)$) = %.5f '%(sample, eff))

## Explanation: efficiency of abs(delta phi(cluster, MET)) < 0.75 and summary of the selections

`integral_low(h, sample, 0.75)` gives the fraction of clusters with |dphi(cluster, MET)| below 0.75. For data in the OOT region this is close to 0.75/pi ~ 0.24, i.e. just the geometric fraction of a flat distribution, which is a nice cross-check that the sample really is uncorrelated pileup; for signal it is much larger because of the peak at 0.

Putting the exercises together, the final cluster selection used by the analysis is: at least one cluster matched to an LLP in the CSC (signal only), `cscRechitClusterJetVetoPt < 10`, `cscRechitClusterMuonVetoPt < 20`, no hits in ME11/ME12, `abs(eta) < 2`, `-12.5 < cluster time < -5` ns for the signal region (`< -12.5` ns for the OOT validation region), and finally the two discriminating variables `N_hits > 130` and `|dphi(cluster, MET)| < 0.75`.

Pitfall: the last two cuts define the corners of the ABCD plane, so they are simultaneously the signal selection and the definition of the sidebands - choosing them from the OOT data control sample (where we are allowed to look) and from the MC signal shape, rather than from the signal region itself, is what keeps the analysis unbiased.